# Exercise 6 : Rearranging Data

Run the first cell : it reads `users.csv` into `users`, the visitors and newsletter signups of a
band's blog per weekday and city. `index_col=0` discards the exported row numbers :

```text
  weekday    city  visitors  signups
0     Sun   Paris       139        7
1     Sun  Berlin       237       12
2     Mon   Paris       326        3
3     Mon  Berlin       456        5
```

In [ ]:
from pathlib import Path

import pandas as pd

DATA = Path.cwd().parents[1] / 'datasets' / 'exercises'
users = pd.read_csv(DATA / 'users.csv', index_col=0)
users

## Step 1 : Pivot

Decide which variable labels the rows (`weekday`), which labels the columns (`city`) and which fills
the cells (`visitors`), and sketch the result on paper.

Pivot `users` with `weekday` as the index, `city` as the columns and `visitors` as the values, and
print the result. Does it match your sketch?

In [ ]:
visitors_pivot = users.pivot(index='weekday', columns='city', values='visitors')
print(visitors_pivot)

## Step 2 : Pivoting all columns

Pivot `users` again with `signups` as the values and print it.

Pivot once more without `values`. Print the result : both `visitors` and `signups` are pivoted,
under two-level column labels. Select the `signups` table from it.

In [ ]:
signups_pivot = users.pivot(index='weekday', columns='city', values='signups')
print(signups_pivot)

pivot = users.pivot(index='weekday', columns='city')
print(pivot)
print(pivot['signups'])

## Step 3 : Unstack and stack

Set the index of `users` to `['city', 'weekday']`. Unstack the `weekday` level into `byweekday`
and print it. Stack `byweekday` by `weekday` and compare the result with the indexed `users`.

Remove the Berlin Sunday row with `drop(index=('Berlin', 'Sun'))` and call the result `sparse`.
Unstack and stack the weekday level of `sparse`. How many rows does the result have, and what is in
the extra row? Why is that row different from zero visitors?

In [ ]:
indexed = users.set_index(['city', 'weekday'])
byweekday = indexed.unstack(level='weekday')
print(byweekday)
print(byweekday.stack(level='weekday'))

sparse = indexed.drop(index=('Berlin', 'Sun'))
expanded = sparse.unstack(level='weekday').stack(level='weekday')
print(expanded)
print(len(sparse), len(expanded))

## Step 4 : The other level

Unstack the `city` level of the indexed `users` into `bycity` and print it. Stack it again by
`city`. Are the rows in the same order as in `users`?

In [ ]:
bycity = indexed.unstack(level='city')
print(bycity)
print(bycity.stack(level='city'))

## Step 5 : Index order

Stack `bycity` back into `newusers`. Swap the two index levels with `.swaplevel(0, 1)` and print
the result. Is the index sorted? Sort it with `sort_index()` and compare it with
`users.sort_index()` using `.equals()`.

In [ ]:
newusers = bycity.stack(level='city')
newusers = newusers.swaplevel(0, 1)
print(newusers)
newusers = newusers.sort_index()
print(newusers)
print(newusers.equals(indexed.sort_index()))

## Step 6 : Melt

Take the visitors table from Step 1 and turn its weekday index back into a column with
`reset_index()`. Melt it with `id_vars='weekday'`, `var_name='city'` and `value_name='visitors'`
into one row per weekday and city. How many rows do you get?

In [ ]:
long = visitors_pivot.reset_index().melt(id_vars='weekday', var_name='city',
                                         value_name='visitors')
print(long)
print(len(long))

## Step 7 : Pivot tables

Read `titanic.csv`. Each passenger has a `pclass` (1, 2 or 3), a `sex` and `survived` (1 or 0).

Compute the survival rate per class and sex with `pivot_table`, with `pclass` as the index, `sex`
as the columns, `survived` as the values and `aggfunc='mean'`. Repeat it with `aggfunc='count'` to
see how many passengers each rate rests on. Why does `pivot` fail on this data?

In [ ]:
titanic = pd.read_csv(DATA / 'titanic.csv')
print(titanic.pivot_table(index='pclass', columns='sex', values='survived', aggfunc='mean'))
print(titanic.pivot_table(index='pclass', columns='sex', values='survived', aggfunc='count'))

## If time permits :

- Read `users.csv` again as in the first cell. Reshape it so that each row is a city and each
  column a weekday, with the signups as values, and add a column `total` with each city's signups

In [ ]:
users = pd.read_csv(DATA / 'users.csv', index_col=0)
signups = users.pivot(index='city', columns='weekday', values='signups')
signups['total'] = signups.sum(axis='columns')
print(signups)